<a href="https://colab.research.google.com/github/ProductPriceTrackerOrg/data-science/blob/main/notebooks/product-matching/02_dataset_creation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Product Matching Model - Dataset creation**

## **Load the dataset**

In [1]:
import pandas as pd
import numpy as np
from itertools import combinations
import random
from collections import defaultdict

# Set random seed for reproducibility
random.seed(42)
np.random.seed(42)

print("Libraries imported successfully!")

Libraries imported successfully!


In [23]:
# Load the dataset
try:
    df = pd.read_csv('/content/pricerunner_aggregate.csv')
    print(f"Dataset loaded successfully!")
    print(f"Dataset shape: {df.shape}")
    print(f"\nDataset columns: {df.columns.tolist()}")
    print(f"\nFirst few rows:")
    print(df.head())

    # Check for required columns
    required_columns = ['Product ID', 'Product Title', 'Cluster ID']
    missing_columns = [col for col in required_columns if col not in df.columns]
    if missing_columns:
        print(f"\nWarning: Missing required columns: {missing_columns}")
    else:
        print(f"\nAll required columns present!")

except FileNotFoundError:
    print("Error: 'your_dataset.csv' file not found. Please ensure the file exists in the current directory.")

Dataset loaded successfully!
Dataset shape: (35311, 7)

Dataset columns: ['Product ID', 'Product Title', 'Vendor ID', 'Cluster ID', 'Cluster Label', 'Category ID', 'Category Label']

First few rows:
   Product ID                                      Product Title  Vendor ID  \
0           1                    apple iphone 8 plus 64gb silver          1   
1           2                apple iphone 8 plus 64 gb spacegrau          2   
2           3  apple mq8n2b/a iphone 8 plus 64gb 5.5 12mp sim...          3   
3           4                apple iphone 8 plus 64gb space grey          4   
4           5  apple iphone 8 plus gold 5.5 64gb 4g unlocked ...          5   

   Cluster ID             Cluster Label  Category ID Category Label  
0           1  Apple iPhone 8 Plus 64GB         2612  Mobile Phones  
1           1  Apple iPhone 8 Plus 64GB         2612  Mobile Phones  
2           1  Apple iPhone 8 Plus 64GB         2612  Mobile Phones  
3           1  Apple iPhone 8 Plus 64GB       

In [24]:
# Display dataset statistics
print("=== Dataset Statistics ===")
print(f"Total products: {len(df)}")
print(f"Unique products: {df['Product Title'].nunique()}")
print(f"Unique clusters: {df['Cluster ID'].nunique()}")

# Check cluster distribution
cluster_counts = df.groupby('Cluster ID').size()
print(f"\nCluster size distribution:")
print(f"Min cluster size: {cluster_counts.min()}")
print(f"Max cluster size: {cluster_counts.max()}")
print(f"Average cluster size: {cluster_counts.mean():.2f}")
print(f"Clusters with >1 product: {sum(cluster_counts > 1)}")

# Check for duplicates
print(f"\nData quality checks:")
print(f"Duplicate Product IDs: {df['Product ID'].duplicated().sum()}")
print(f"Missing values in Product ID: {df['Product ID'].isnull().sum()}")
print(f"Missing values in Product Title: {df['Product Title'].isnull().sum()}")
print(f"Missing values in Cluster ID: {df['Cluster ID'].isnull().sum()}")

=== Dataset Statistics ===
Total products: 35311
Unique products: 30993
Unique clusters: 13233

Cluster size distribution:
Min cluster size: 1
Max cluster size: 27
Average cluster size: 2.67
Clusters with >1 product: 7859

Data quality checks:
Duplicate Product IDs: 0
Missing values in Product ID: 0
Missing values in Product Title: 0
Missing values in Cluster ID: 0


In [25]:
# remove the duplicated product titles
df = df.drop_duplicates(subset=['Product Title'])

# Display dataset statistics
print("=== Dataset Statistics ===")
print(f"Total products: {len(df)}")
print(f"Unique products: {df['Product Title'].nunique()}")
print(f"Unique clusters: {df['Cluster ID'].nunique()}")


=== Dataset Statistics ===
Total products: 30993
Unique products: 30993
Unique clusters: 12885


## **Generate positive pairs (matches)**

In [26]:
def generate_positive_pairs(dataframe):
    """
    Generate positive pairs (matches) from products in the same cluster.

    Args:
        dataframe: Input DataFrame with Product ID and Cluster ID columns

    Returns:
        DataFrame with columns: id1, id2, match
    """
    positive_pairs = []

    # Group by Cluster ID
    clustered_groups = dataframe.groupby('Cluster ID')

    for cluster_id, group in clustered_groups:
        # Only process clusters with more than one product
        if len(group) > 1:
            product_ids = group['Product ID'].tolist()

            # Generate all possible unique pairs within this cluster
            for id1, id2 in combinations(product_ids, 2):
                positive_pairs.append({
                    'id1': id1,
                    'id2': id2,
                    'match': 1
                })

    positive_df = pd.DataFrame(positive_pairs)
    return positive_df

# Generate positive pairs
print("Generating positive pairs...")
positive_pairs_df = generate_positive_pairs(df)

print(f"Generated {len(positive_pairs_df)} positive pairs")
print(f"Sample positive pairs:")
print(positive_pairs_df.head())

# check duplicates
print(f"\nData quality checks:")
print(f"Duplicate positive pairs: {positive_pairs_df[['id1', 'id2']].duplicated().sum()}")

Generating positive pairs...
Generated 51441 positive pairs
Sample positive pairs:
   id1  id2  match
0    1    2      1
1    1    3      1
2    1    4      1
3    1    5      1
4    1    7      1

Data quality checks:
Duplicate positive pairs: 0


## **Generate negative pairs (no-matches)**

In [27]:
def generate_negative_pairs(dataframe, num_negative_pairs):
    """
    Generate negative pairs (no-matches) from products in different clusters.

    Args:
        dataframe: Input DataFrame with Product ID and Cluster ID columns
        num_negative_pairs: Number of negative pairs to generate

    Returns:
        DataFrame with columns: id1, id2, match
    """
    # Create mapping of Product ID to Cluster ID for efficient lookup
    product_to_cluster = dict(zip(dataframe['Product ID'], dataframe['Cluster ID']))
    all_product_ids = list(dataframe['Product ID'].unique())

    negative_pairs = []
    processed_pairs = set()  # Use a set for O(1) duplicate checking
    attempts = 0
    max_attempts = num_negative_pairs * 10  # Prevent infinite loop

    while len(negative_pairs) < num_negative_pairs and attempts < max_attempts:
        attempts += 1

        # Randomly select two Product IDs
        id1, id2 = random.sample(all_product_ids, 2)

        # Check if they belong to different clusters
        if product_to_cluster[id1] != product_to_cluster[id2]:
            # Create a sorted tuple to act as a unique key for the pair
            # This handles both (id1, id2) and (id2, id1) as the same pair
            pair_key = tuple(sorted((id1, id2)))

            # Check if this pair has already been added (O(1) operation)
            if pair_key not in processed_pairs:
                negative_pairs.append({
                    'id1': id1,
                    'id2': id2,
                    'match': 0
                })
                processed_pairs.add(pair_key)

    if len(negative_pairs) < num_negative_pairs:
        print(f"Warning: Could only generate {len(negative_pairs)} negative pairs out of {num_negative_pairs} requested")

    print(f"Generated pairs in {attempts} attempts (efficiency: {len(negative_pairs)/attempts*100:.1f}%)")

    return pd.DataFrame(negative_pairs)

# Generate negative pairs equal to the number of positive pairs
print("Generating negative pairs...")
target_negative_pairs = len(positive_pairs_df)
negative_pairs_df = generate_negative_pairs(df, target_negative_pairs)

print(f"Generated {len(negative_pairs_df)} negative pairs")
print(f"Target was {target_negative_pairs} negative pairs")
print(f"Sample negative pairs:")
print(negative_pairs_df.head())

# check duplicates
print(f"\nData quality checks:")
print(f"Duplicate negative pairs: {negative_pairs_df[['id1', 'id2']].duplicated().sum()}")

Generating negative pairs...
Generated pairs in 51450 attempts (efficiency: 100.0%)
Generated 51441 negative pairs
Target was 51441 negative pairs
Sample negative pairs:
     id1    id2  match
0  13722  46091      0
1  22118  38954      0
2  32659   2915      0
3   1174  42249      0
4  37602  11018      0

Data quality checks:
Duplicate negative pairs: 0


In [28]:
def combine_and_finalize(positive_df, negative_df, original_df):
    """
    Combine positive and negative pairs, merge with original data for titles, and shuffle.

    Args:
        positive_df: DataFrame with positive pairs
        negative_df: DataFrame with negative pairs
        original_df: Original dataset with Product ID and Product Title

    Returns:
        Final DataFrame with columns: title1, title2, match
    """
    # Combine positive and negative pairs
    combined_df = pd.concat([positive_df, negative_df], ignore_index=True)
    print(f"Combined dataset shape: {combined_df.shape}")

    # Create mapping of Product ID to Product Title
    id_to_title = dict(zip(original_df['Product ID'], original_df['Product Title']))

    # Add title1 and title2 columns
    combined_df['title1'] = combined_df['id1'].map(id_to_title)
    combined_df['title2'] = combined_df['id2'].map(id_to_title)

    # Select final columns and shuffle
    final_df = combined_df[['title1', 'title2', 'match']].copy()
    final_df = final_df.sample(frac=1, random_state=42).reset_index(drop=True)

    return final_df

# Combine and finalize the dataset
print("Combining positive and negative pairs...")
final_training_df = combine_and_finalize(positive_pairs_df, negative_pairs_df, df)

print(f"Final dataset shape: {final_training_df.shape}")
print(f"Match distribution:")
print(final_training_df['match'].value_counts().sort_index())

Combining positive and negative pairs...
Combined dataset shape: (102882, 3)
Final dataset shape: (102882, 3)
Match distribution:
match
0    51441
1    51441
Name: count, dtype: int64


## **Save the final dataset**

In [29]:
# Save the final dataset
output_filename = 'training_pairs.csv'
final_training_df.to_csv(output_filename, index=False)
print(f"Training dataset saved to '{output_filename}'")

# Display final summary
print("\n" + "="*50)
print("FINAL DATASET SUMMARY")
print("="*50)
print(f"Total training pairs created: {len(final_training_df)}")
print(f"Positive pairs (matches): {sum(final_training_df['match'] == 1)}")
print(f"Negative pairs (no-matches): {sum(final_training_df['match'] == 0)}")
print(f"Dataset balance: {(final_training_df['match'].value_counts(normalize=True) * 100).round(2)}%")

print(f"\nFirst 5 rows of the final dataset:")
print(final_training_df.head())

print(f"\nDataset saved successfully to: {output_filename}")

Training dataset saved to 'training_pairs.csv'

FINAL DATASET SUMMARY
Total training pairs created: 102882
Positive pairs (matches): 51441
Negative pairs (no-matches): 51441
Dataset balance: match
1    50.0
0    50.0
Name: proportion, dtype: float64%

First 5 rows of the final dataset:
                                              title1  \
0                    bosch kgn49xl30g fridge freezer   
1        liebherr ctpwb2121 55cm fridge freezer blue   
2  candy cvs1482d3 1400rpm 8kg smart touch washin...   
3  55sk8100 55 inch ips 4k nano cell super uhd hd...   
4  panasonic tx 49fx750b 49 4k ultra hd hdr led s...   

                                              title2  match  
0  bosch kgn49xl30g 70cm serie 4 frost free fridg...      1  
1  liebherr k hl /gefrier kombination mit smartfr...      1  
2     aeg sks58240f0 built under fridge with ice box      0  
3  hoover hvtu542bhk undercounter freestanding fr...      0  
4  panasonic tx 49fx750b 49 smart 4k ultra hd hdr...      1  

Dat

In [30]:
# Additional validation
print("\n" + "="*50)
print("DATA VALIDATION")
print("="*50)

# Check for missing values
print("Missing values check:")
print(final_training_df.isnull().sum())

# Check for duplicate pairs
print(f"\nDuplicate pairs check:")
# Create a sorted pair representation for duplicate detection
final_training_df['pair_key'] = final_training_df.apply(
    lambda row: tuple(sorted([row['title1'], row['title2']])), axis=1
)
duplicates = final_training_df['pair_key'].duplicated().sum()
print(f"Duplicate pairs found: {duplicates}")

# Clean up the temporary column
final_training_df = final_training_df.drop('pair_key', axis=1)

# Verify no positive pairs are actually negative (same cluster check)
print(f"\nData integrity check completed!")
print(f"Ready for Siamese Network training!")


DATA VALIDATION
Missing values check:
title1    0
title2    0
match     0
dtype: int64

Duplicate pairs check:
Duplicate pairs found: 0

Data integrity check completed!
Ready for Siamese Network training!
